# Running on Modal

## What you'll learn

- Write a **tool op** — `ToolSpec` + `build_command()` instead of `execute()`
- Deploy it once as a persistent Modal endpoint with `artisan modal deploy`
- Route steps to the endpoint by flipping `compute_provider="modal"`
- Understand the spawn/poll job model and per-artifact fan-out
- Authenticate clients with Modal proxy-auth tokens
- Debug endpoint execution

**Prerequisites:** [First Pipeline](../01-getting-started/01-first-pipeline.ipynb),
[Compute Routing](01-compute-routing.ipynb).
**Estimated time:** 15 minutes
**GPU required:** No (the demo tool runs on CPU containers).

:::{note}
This tutorial requires a Modal account, Modal credentials, and a deployed
endpoint. Code cells are shown for reference and are not executed in the
docs build.
:::

In [ ]:
from __future__ import annotations

from artisan.operations.examples import EchoTool
from artisan.orchestration import PipelineManager
from artisan.schemas.operation_config.compute import ComputeProvider, ModalComputeConfig
from artisan.utils import tutorial_setup
from artisan.visualization import inspect_pipeline

In [ ]:
env = tutorial_setup("modal_execution")
DELTA_ROOT = env.delta_root

## Tool operations

Modal compute runs **tool ops**: operations that wrap an external tool. A
tool op declares a `ToolSpec`, a `Params` model, and a `build_command()` —
and **no** `execute()`; the framework provides one:

```python
class EchoTool(OperationDefinition):
    name = "echo_tool"
    tool = ToolSpec(executable="bash", interpreter=None)
    compute_provider = ComputeProvider(modal=ModalComputeConfig())

    class Params(BaseModel):
        model_config = {"extra": "forbid"}   # the endpoint's typed schema

        text: str = Field(default="hello from echo_tool", description="...")
        filename: str = Field(default="echo.txt", description="...")

    params: Params = Params()

    def build_command(self, inputs: dict[str, Any]) -> list[str]:
        return [
            *self.tool.parts(),
            "-c",
            f'printf "%s\\n" "{self.params.text}" > "{self.params.filename}"',
        ]
```

`build_command` is the single source of the command. Under
`compute_provider="local"` the framework runs it as a local subprocess;
under `"modal"` the deployed copy runs it in the tool's container. A tool
op's products are the files its command writes to the execute dir — memory
results and post-run glue belong in `postprocess()`.

Pure-Python operations (custom `execute()`) run on the local provider only.

## Deploy the endpoint (once per tool)

```bash
artisan modal deploy echo_tool
```

This deploys one persistent Modal app named `artisan-tool-echo_tool`: a
**worker** (the tool's image + hardware, one job per container) behind a
lightweight **HTTP endpoint** with a typed, tool-native API:

| Route | Method | Purpose |
|-------|--------|---------|
| `/submit` | POST | `Params` JSON + input files (multipart) → `call_id` |
| `/result` | GET | Poll job status; returns the output manifest |
| `/download` | GET | Stream the output files as a tar |
| `/cancel` | POST | Terminate the running container |
| `/docs` | GET | Swagger UI |

Deploy reads the op's **class-level** config: image, volumes, secrets, and
scaling from `ModalComputeConfig`; gpu/cpu/memory/timeout from
`ComputeResources`. Redeploy after changing the op's code, image, or
hardware.

The same endpoint serves Artisan pipelines and non-Artisan callers (curl,
other repositories) alike.

## Authentication

The endpoint requires Modal **proxy-auth tokens** (created in the Modal
dashboard under *Settings → Proxy Auth Tokens*). Clients send them as
`Modal-Key` / `Modal-Secret` headers.

The recommended setup is a `.env` file at the repo root — copy the
committed `.env.example` and fill in your token:

```bash
cp .env.example .env   # .env is gitignored
# then edit:
#   MODAL_PROXY_TOKEN_ID=wk-...
#   MODAL_PROXY_TOKEN_SECRET=ws-...
```

Artisan discovers the tokens from the process environment first, then the
nearest `.env` file walking up from the working directory — so Jupyter
kernels, cron jobs, and IDE test runners all work without shell-inherited
exports. Environment variables of the same names override the file (CI).

A different variable prefix can be configured per op via
`ModalComputeConfig.auth_secret`.

## Hardware lives on the deployment

Worker hardware is read from the op's class-level `ComputeResources` at
deploy time:

```python
class RFdiffusion(OperationDefinition):
    ...
    compute_provider = ComputeProvider(modal=ModalComputeConfig(image=RFD_IMAGE))
    compute_resources = ComputeResources(gpu="A100", memory_gb=32, timeout=7200)
```

Changing hardware means redeploying — a per-step `compute_resources`
override does **not** reconfigure an already-deployed endpoint.

In [ ]:
config = ComputeProvider(active="modal", modal=ModalComputeConfig())

print(f"Active provider: {config.active}")
print(f"Available:       {config.available()}")
print(f"Worker image:    {config.modal.image}")
print(f"Poll interval:   {config.modal.poll_interval}s")

## Running a step on the endpoint

`compute_provider="modal"` is the only change — the operation, params, and
output wiring stay identical:

In [ ]:
pipeline = PipelineManager.create(
    name="modal_tutorial",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
    default_compute_provider="local",
)

# local first — same op, no Modal required
pipeline.run(
    operation=EchoTool,
    name="echo_local",
    params={"text": "ran locally", "filename": "local.txt"},
)

# then on the deployed endpoint — one argument changed
pipeline.run(
    operation=EchoTool,
    name="echo_modal",
    params={"text": "ran on Modal", "filename": "modal.txt"},
    compute_provider="modal",
)

summary = pipeline.finalize()
print(f"Pipeline complete: success={summary['overall_success']}")
inspect_pipeline(DELTA_ROOT)

## What happens during a modal step

For each artifact, the framework's `execute()`:

1. **Submits** the op's `Params` + input files to `/submit` (inline
   multipart; inputs that already live on object storage pass their
   `s3://` URI with zero re-upload)
2. **Polls** `/result` at `poll_interval` until the job leaves `pending`
3. **Downloads** the output tar into the artifact's `execute_dir` —
   recreating the exact layout of a local run (the tool log arrives
   separately and lands in the unit log, as it does locally)
4. Hands off to `postprocess`, lineage capture, and recording — the
   lifecycle is identical to local execution

A unit of N artifacts dispatches as N concurrent endpoint calls; the
worker runs one job per container (`max_inputs=1`), so Modal scales by
adding containers. Pipeline cancellation posts `/cancel`, which terminates
the running containers. The tool log arrives with the result (manifest
tail + tar), not streamed live.

**Transport limits:** inline inputs and the output tar are bounded at
100 MB per direction. Larger inputs should be `s3://` URIs; large static
data (model weights) belongs on Modal Volumes
(`ModalComputeConfig.volumes`), not in the request.

## Calling the endpoint without Artisan

The endpoint is a plain HTTP API — Artisan is one client among others:

```bash
URL=https://<workspace>--artisan-tool-echo-tool.modal.run

# submit
curl -X POST "$URL/submit" \
  -H "Modal-Key: $MODAL_PROXY_TOKEN_ID" \
  -H "Modal-Secret: $MODAL_PROXY_TOKEN_SECRET" \
  -F 'params={"text": "hi from curl", "filename": "out.txt"}'
# → {"call_id": "fc-..."}

# poll
curl "$URL/result?call_id=fc-..." \
  -H "Modal-Key: $MODAL_PROXY_TOKEN_ID" -H "Modal-Secret: $MODAL_PROXY_TOKEN_SECRET"

# download outputs
curl -o outputs.tar "$URL/download?call_id=fc-..." \
  -H "Modal-Key: $MODAL_PROXY_TOKEN_ID" -H "Modal-Secret: $MODAL_PROXY_TOKEN_SECRET"
```

## Debugging

| Problem | Cause | Fix |
|---------|-------|-----|
| `tool_endpoint_misconfigured` | Op is not a tool op, or modal config missing | Declare `ToolSpec` + `build_command()`; configure `compute_provider.modal` |
| "has no web URL — is it deployed?" | Endpoint not deployed | `artisan modal deploy <op>` |
| HTTP 407/401 at submit | Missing or invalid proxy-auth tokens | Set `MODAL_PROXY_TOKEN_ID` / `MODAL_PROXY_TOKEN_SECRET` |
| HTTP 422 at submit | Params don't match the op's schema | The endpoint validates against `Params` (`extra="forbid"`) |
| `op_execute_failed` | The tool exited non-zero | The error carries the stderr tail; full log in the parquet `tool_output` column |
| Result `expired` | Output polled more than 7 days after completion | Re-run the step |

Develop locally first: run with `compute_provider="local"` until the
pipeline logic is correct, then flip to `"modal"`. Same op, same
`build_command`, same outputs.

## Summary

| Concept | What it does |
|---------|-------------|
| Tool op | `ToolSpec` + `Params` + `build_command()`; no `execute()` |
| `artisan modal deploy <op>` | One-time deploy: persistent worker + HTTP endpoint per tool |
| `compute_provider="modal"` | Route a step's per-artifact execute() calls to the endpoint |
| Spawn/poll | `/submit` → `call_id`; poll `/result`; `/download` outputs |
| `ComputeResources` | Worker hardware, read at deploy time |
| Proxy auth | `MODAL_PROXY_TOKEN_ID` / `MODAL_PROXY_TOKEN_SECRET` headers |
| Inline transport | ≤100 MB per direction; `s3://` URIs bypass the bound |

Operations, inputs, params, and output wiring are identical whether the
tool runs locally or on Modal. Results land in the same Delta Lake tables
regardless.

## Next steps

- [Compute Routing](01-compute-routing.ipynb) — Step runners vs compute providers
- [SLURM Execution](02-slurm-execution.ipynb) — Run operations on a SLURM cluster
- [Configure Execution](../../how-to-guides/configuring-execution.md) — Complete configuration reference
- [Execution Flow](../../concepts/execution-flow.md) — How the framework dispatches and tracks work